In [ ]:
#This notebook evaluates the effect of the wealth index using statistical t-tests to get Table 11
import os
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from sklearn.metrics import accuracy_score, matthews_corrcoef
from scipy.stats import ttest_rel


N_SPLITS = 10
LABEL = "EnergyPoor"

In [2]:
INCLUSIVE_PATH = r"PAKISTAN\DATA\PAIRTEST10\WEALTH_INCLUSIVE\MODELS"
EXCLUSIVE_PATH = r"PAKISTAN\DATA\PAIRTEST10\WEALTH_EXCLUSIVE\MODELS"

TEST_DATA_PATH = r"PAKISTAN\DATA\PAIRTEST10"

inc_acc, inc_mcc = [], []
exc_acc, exc_mcc = [], []

BEST_INCLUSIVE = "DL_NN_TORCH"
BEST_EXCLUSIVE = "TRAD_GBM"

RESULTS_FOLDER = r"PAKISTAN\RESULT"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

for i in range(1, N_SPLITS + 1):

    # ---------------- LOAD MODELS ----------------
    inc_model_path = f"{INCLUSIVE_PATH}/split{i}/{BEST_INCLUSIVE}"
    exc_model_path = f"{EXCLUSIVE_PATH}/split{i}/{BEST_EXCLUSIVE}"

    inc_model = TabularPredictor.load(inc_model_path)
    exc_model = TabularPredictor.load(exc_model_path)

    # ---------------- LOAD TEST DATA ----------------
    inc_test = pd.read_parquet(f"{TEST_DATA_PATH}/split{i}_test.parquet")
    exc_test = pd.read_parquet(f"{TEST_DATA_PATH}/split{i}_test.parquet")

    # ---------------- SPLIT FEATURES / LABEL ----------------
    y_inc = inc_test[LABEL]
    X_inc = inc_test.drop(columns=[LABEL])

    y_exc = exc_test[LABEL]
    X_exc = exc_test.drop(columns=[LABEL])

    # ---------------- PREDICTIONS ----------------
    pred_inc = inc_model.predict(X_inc)
    pred_exc = exc_model.predict(X_exc)

    # ---------------- METRICS ----------------
    inc_acc.append(accuracy_score(y_inc, pred_inc))
    inc_mcc.append(matthews_corrcoef(y_inc, pred_inc))

    exc_acc.append(accuracy_score(y_exc, pred_exc))
    exc_mcc.append(matthews_corrcoef(y_exc, pred_exc))

    print(f"Split {i}: Acc Inc={inc_acc[-1]:.3f}, Acc Exc={exc_acc[-1]:.3f}")
   

Split 1: Acc Inc=0.880, Acc Exc=0.807
Split 2: Acc Inc=0.880, Acc Exc=0.818
Split 3: Acc Inc=0.870, Acc Exc=0.795
Split 4: Acc Inc=0.882, Acc Exc=0.801
Split 5: Acc Inc=0.891, Acc Exc=0.812
Split 6: Acc Inc=0.882, Acc Exc=0.806
Split 7: Acc Inc=0.888, Acc Exc=0.808
Split 8: Acc Inc=0.887, Acc Exc=0.809
Split 9: Acc Inc=0.873, Acc Exc=0.795
Split 10: Acc Inc=0.888, Acc Exc=0.813


In [3]:
import numpy as np
from scipy.stats import ttest_rel
import pandas as pd

# ---------------- CONVERT AFTER LOOP ----------------
inc_acc = np.array(inc_acc)
exc_acc = np.array(exc_acc)

inc_mcc = np.array(inc_mcc)
exc_mcc = np.array(exc_mcc)

# ---------------- T-TEST ----------------
diff_acc = inc_acc - exc_acc
t_acc, p_acc = ttest_rel(inc_acc, exc_acc)

diff_mcc = inc_mcc - exc_mcc
t_mcc, p_mcc = ttest_rel(inc_mcc, exc_mcc)

# ---------------- FINAL TABLE ----------------
df_result = pd.DataFrame([
    {
        "Country": "Pakistan",
        "Comparison": "Inclusive vs Exclusive",
        "Metric": "Accuracy",
        "Δ (Mean)": diff_acc.mean(),
        "SD": diff_acc.std(ddof=1),
        "t-value": t_acc,
        "p-value": p_acc
    },
    {
        "Country": "Pakistan",
        "Comparison": "Inclusive vs Exclusive",
        "Metric": "MCC",
        "Δ (Mean)": diff_mcc.mean(),
        "SD": diff_mcc.std(ddof=1),
        "t-value": t_mcc,
        "p-value": p_mcc
    }
])

df_result

,Country,Comparison,Metric,Δ (Mean),SD,t-value,p-value
0,Pakistan,Inclusive vs Exclusive,Accuracy,0.075562,0.005536,43.163127,9.600470e-12
1,Pakistan,Inclusive vs Exclusive,MCC,0.155088,0.011153,43.973579,8.126326e-12


In [4]:

# ---------- Save after inspection ----------
save_confirm = True   # change to False if don't want to save yet

if save_confirm:
    df_result.to_csv(rf"{RESULTS_FOLDER}\Table11_pakistan_wealth_effect_ttest_results.csv", index=False)
    print("\nFiles saved successfully.")


Files saved successfully.
